# Example 10.3 / 1: Rowwise and columnwise equivalents

Companion notebook for **10.3 Columnwise Formulations** in the AMPL lab Documentation, section "Rowwise and columnwise equivalents". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Rowwise and columnwise equivalents

Its columnwise equivalent is:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PRODUCTS;
param profit {PRODUCTS};
param hours {PRODUCTS} >= 0;
param available >= 0;

maximize TotalProfit;
subject to TimeLimit: to_come <= available;

var Make {p in PRODUCTS} >= 0,
    obj TotalProfit profit[p],
    coeff TimeLimit hours[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PRODUCTS"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["hours"] = {"bands": 0.5, "coils": 1.0}
ampl.param["available"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.obj["TotalProfit"].value())

HiGHS 1.14.0: optimal solution; objective 2000
0 simplex iterations
0 barrier iterations
solved
2000.0


### Pure AMPL: data.dat and model.run

The same columnwise model with pure-AMPL data:

In [6]:
%%writefile data.dat
set PRODUCTS := bands coils;
param profit := bands 25 coils 30;
param hours := bands 0.5 coils 1;
param available := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display TotalProfit;

HiGHS 1.14.0: optimal solution; objective 2000
0 simplex iterations
0 barrier iterations
TotalProfit = 2000

